## Imports

In [1]:
import os
import re
import time
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 50)

In [3]:
## Connection
connection = presto.connect(
        
        host='presto-gateway.processing.data.production.internal',
#     presto.processing.yoda.run
#     presto-gateway.processing.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [4]:
cwd = os.getcwd()
print(cwd)
local_datasource = '/Users/E2074/local-datasets/customer/ads-monetisation/oneline-sale'
print(local_datasource)

/Users/E2074/analytics/customer/Ads-Monetisation/experiments/onlinesale
/Users/E2074/local-datasets/customer/ads-monetisation/oneline-sale


## Datasets & Parameter

In [5]:
## Parameter 
current_date = datetime.now()
start_date = '20240820'
end_date = '20240905'

city = 'Bangalore'

# Convert date strings to datetime objects
start_dt = datetime.strptime(start_date, '%Y%m%d')
end_dt = datetime.strptime(end_date, '%Y%m%d')
segment_date = end_dt.strftime('%Y-%m-%d')
print(start_date, ' to ' ,end_date)

20240820  to  20240905


### Gross Customers

In [6]:
data_query = f"""

WITH customer_base as (
    
    SELECT 
        customer_id,
        city_name,
        customer_obj_mobile customer_mobile,
        
        COUNT(DISTINCT order_id) gross_count,
        COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
        COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
        COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
        COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara

    FROM    
        orders.order_logs_fact
    WHERE  
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND city_name = 'Bangalore'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','Bike Metro','CabEconomy','CabPremium')
    GROUP BY 1,2,3
    ),
    
    segment AS (
    
    SELECT
        customer_id,
        taxi_ltr_segment ltr_segment,
        taxi_retention_segments retention_segment,
        taxi_lifetime_stage lifetime_stage,
        taxi_service_affinity service_affinity,
        gender_tag gender
    FROM
        datasets.iallocator_customer_segments
    WHERE 
        run_date = '{segment_date}'
    )
    
    SELECT 
        customer_base.*,
        coalesce( ltr_segment, 'UNKNOWN')ltr_segment,
        coalesce( retention_segment, 'UNKNOWN') retention_segment,
        coalesce( lifetime_stage, 'UNKNOWN') lifetime_stage,
        coalesce( service_affinity, 'UNKNOWN') service_affinity,
        coalesce( gender, 'UNKNOWN') gender
    FROM 
        customer_base
    LEFT JOIN 
        segment
        ON customer_base.customer_id = segment.customer_id
    WHERE 
        LENGTH(customer_mobile) = 10
"""

# Execute the query and get the result as a DataFrame
df_data = pd.read_sql(data_query, connection)

df_data.to_csv( local_datasource + '/customer_data_{}_{}.csv'.format(start_date, end_date), index=False)

In [ ]:
# df_data = pd.read_csv( local_datasource + '/analysis_data_{}_{}.csv'.format(start_date, end_date))

In [7]:
df_data.head(5)

,customer_id,city_name,customer_mobile,gross_count,net_count,cobrm,cobra,ocara,ltr_segment,retention_segment,lifetime_stage,service_affinity,gender
0,665efd9c9cdceb765a16d9f7,Bangalore,8951210983,7,3,0,0,0,PHH,GOLD,HOOK,LINK_AUTO,MALE
1,5bc5cdcbee78b247336513f9,Bangalore,9886766470,5,5,0,0,0,PHH,PLATINUM,HOOK,ONLY_AUTO,MALE
2,632afadf06043f3ecbc98d7a,Bangalore,9986964622,9,6,0,3,0,PHH,PLATINUM,COMMITTED,ONLY_AUTO,UNKNOWN
3,62711d983833b62bc2c8af52,Bangalore,9821174269,55,17,0,14,14,PHH,ELITE,COMMITTED,ONLY_AUTO,FEMALE
4,61658667fbce66506396cb42,Bangalore,7995245173,1,1,0,0,0,PHH,SILVER,SOFT_CHURN,UNKNOWN,MALE


## User defined function

In [8]:
agg_dict = {
    'customers': ('customer_id', 'nunique'),
    'gross_count': ('gross_count', 'sum'),
    'net_count': ('net_count', 'sum'),
    'cobrm': ('cobrm', 'sum'),
    'cobra': ('cobra', 'sum'),
    'ocara': ('ocara', 'sum'),
}

def calculate_percentage(df, numerator, denominator):
    
    percentage = (df[numerator] * 100.00 / df[denominator]).round(2)
    
    return percentage

def calculate_percentage_columns(df):
        
    df['g2n'] = calculate_percentage(df, 'net_count', 'gross_count')
    df['cobrm %'] = calculate_percentage(df, 'cobrm', 'gross_count')
    df['cobra %'] = calculate_percentage(df, 'cobra', 'gross_count')
    df['ocara %'] = calculate_percentage(df, 'ocara', 'gross_count')
    
    return df

## Sampling

In [9]:
df_data.head(2)

,customer_id,city_name,customer_mobile,gross_count,net_count,cobrm,cobra,ocara,ltr_segment,retention_segment,lifetime_stage,service_affinity,gender
0,665efd9c9cdceb765a16d9f7,Bangalore,8951210983,7,3,0,0,0,PHH,GOLD,HOOK,LINK_AUTO,MALE
1,5bc5cdcbee78b247336513f9,Bangalore,9886766470,5,5,0,0,0,PHH,PLATINUM,HOOK,ONLY_AUTO,MALE


In [10]:
df_data.columns

Index(['customer_id', 'city_name', 'customer_mobile', 'gross_count',
       'net_count', 'cobrm', 'cobra', 'ocara', 'ltr_segment',
       'retention_segment', 'lifetime_stage', 'service_affinity', 'gender'],
      dtype='object')

In [11]:
df_analysis2 = df_data\
.groupby(['city_name'])\
.agg(**agg_dict)\
.reset_index()

df_analysis2 = calculate_percentage_columns(df_analysis2)
df_analysis2

,city_name,customers,gross_count,net_count,cobrm,cobra,ocara,g2n,cobrm %,cobra %,ocara %
0,Bangalore,2063504,11834718,6099772,116096,2443797,1901494,51.54,0.98,20.65,16.07


In [12]:
df_data['city_name'].value_counts(True)

city_name
Bangalore    1.0
Name: proportion, dtype: float64

In [13]:
df_data.shape

(2063523, 13)

In [14]:
df_data.customer_id.nunique()

2063504

In [15]:
duplicated_values = df_data['customer_id'].duplicated(keep=False) 
df_cleaned = df_data[~duplicated_values]
df_cleaned.shape

(2063485, 13)

In [16]:
df_cleaned.customer_id.nunique()

2063485

In [17]:
random_seed = 42

df_train_control, df_test = train_test_split(df_cleaned, 
                                             test_size=0.3, 
                                             random_state=random_seed
                                            )

df_train_control = df_train_control.sample(len(df_train_control), random_state=random_seed)
df_test = df_test.sample(len(df_test), random_state=random_seed)

# Assign group labels
df_train_control['group_tc'] = 'control'
df_test['group_tc'] = 'test'

df_final_sample = pd.concat([df_train_control, df_test])

In [18]:
df_final_sample['group_tc'].value_counts(True)

group_tc
control    0.7
test       0.3
Name: proportion, dtype: float64

In [19]:
df_final_sample.groupby('city_name')['group_tc'].value_counts(True).unstack(fill_value=0)

group_tc,control,test
city_name,,
Bangalore,0.7,0.3


In [20]:
df_final_sample.columns

Index(['customer_id', 'city_name', 'customer_mobile', 'gross_count',
       'net_count', 'cobrm', 'cobra', 'ocara', 'ltr_segment',
       'retention_segment', 'lifetime_stage', 'service_affinity', 'gender',
       'group_tc'],
      dtype='object')

In [21]:
df1 = df_final_sample\
.groupby(['city_name','group_tc'])\
.agg(**agg_dict)\
.reset_index()

df1 = calculate_percentage_columns(df1)
df1

,city_name,group_tc,customers,gross_count,net_count,cobrm,cobra,ocara,g2n,cobrm %,cobra %,ocara %
0,Bangalore,control,1444439,8276604,4270687,80501,1707224,1328484,51.60,0.97,20.63,16.05
1,Bangalore,test,619046,3557957,1828998,35593,736551,572981,51.41,1.00,20.70,16.10


In [22]:
df2 = df_final_sample\
.groupby(['city_name','ltr_segment', 'group_tc'])\
.agg(**agg_dict)\
.reset_index()

df2 = calculate_percentage_columns(df2)
df2

,city_name,ltr_segment,group_tc,customers,gross_count,net_count,cobrm,cobra,ocara,g2n,cobrm %,cobra %,ocara %
0,Bangalore,HH,control,219353,500157,245930,6327,96174,90069,49.17,1.27,19.23,18.01
1,Bangalore,HH,test,94046,216508,105574,2811,42395,39198,48.76,1.30,19.58,18.10
2,Bangalore,PHH,control,1196128,7709835,4024757,69623,1592835,1209340,52.20,0.90,20.66,15.69
3,Bangalore,PHH,test,512420,3312202,1723424,30585,686240,521296,52.03,0.92,20.72,15.74
4,Bangalore,UNKNOWN,control,28958,66612,0,4551,18215,29075,0.00,6.83,27.34,43.65
5,Bangalore,UNKNOWN,test,12580,29247,0,2197,7916,12487,0.00,7.51,27.07,42.69


In [23]:
df3 = df_final_sample\
.groupby(['city_name','retention_segment', 'group_tc'])\
.agg(**agg_dict)\
.reset_index()

df3 = calculate_percentage_columns(df3)
df3

,city_name,retention_segment,group_tc,customers,gross_count,net_count,cobrm,cobra,ocara,g2n,cobrm %,cobra %,ocara %
0,Bangalore,DORMANT,control,113814,237782,1,4650,93577,81564,0.00,1.96,39.35,34.30
1,Bangalore,DORMANT,test,49310,103166,0,1907,40804,35392,0.00,1.85,39.55,34.31
2,Bangalore,ELITE,control,170057,3116947,1683332,26548,628620,480834,54.01,0.85,20.17,15.43
3,Bangalore,ELITE,test,72711,1339392,721153,11891,269231,207317,53.84,0.89,20.10,15.48
4,Bangalore,GOLD,control,341884,1436361,783130,12775,282162,211258,54.52,0.89,19.64,14.71
5,Bangalore,GOLD,test,145902,614038,334855,5569,119809,90665,54.53,0.91,19.51,14.77
6,Bangalore,HH,control,180746,424440,245930,4317,69953,61453,57.94,1.02,16.48,14.48
7,Bangalore,HH,test,77392,183473,105574,1985,30950,26569,57.54,1.08,16.87,14.48
8,Bangalore,INACTIVE,control,43566,80032,0,1716,29621,28792,0.00,2.14,37.01,35.98
9,Bangalore,INACTIVE,test,18860,34502,0,700,12611,12513,0.00,2.03,36.55,36.27


In [24]:
df4 = df_final_sample\
.groupby(['city_name','lifetime_stage', 'group_tc'])\
.agg(**agg_dict)\
.reset_index()

df4 = calculate_percentage_columns(df4)
df4

,city_name,lifetime_stage,group_tc,customers,gross_count,net_count,cobrm,cobra,ocara,g2n,cobrm %,cobra %,ocara %
0,Bangalore,CHURN_OTB,control,76713,205957,123043,1848,36496,27638,59.74,0.90,17.72,13.42
1,Bangalore,CHURN_OTB,test,32834,88879,52479,788,16343,12134,59.05,0.89,18.39,13.65
2,Bangalore,COMMITTED,control,349847,4749366,2512788,41731,970983,735366,52.91,0.88,20.44,15.48
3,Bangalore,COMMITTED,test,149945,2040820,1077271,18460,418028,317145,52.79,0.90,20.48,15.54
4,Bangalore,DETOX,control,1763,10111,4187,96,2904,1387,41.41,0.95,28.72,13.72
5,Bangalore,DETOX,test,735,4163,1879,29,1135,541,45.14,0.70,27.26,13.00
6,Bangalore,DORMANT,control,113814,237782,1,4650,93577,81564,0.00,1.96,39.35,34.30
7,Bangalore,DORMANT,test,49310,103166,0,1907,40804,35392,0.00,1.85,39.55,34.31
8,Bangalore,HANDHOLDING,control,180746,424440,245930,4317,69953,61453,57.94,1.02,16.48,14.48
9,Bangalore,HANDHOLDING,test,77392,183473,105574,1985,30950,26569,57.54,1.08,16.87,14.48


In [25]:
df_final_sample.groupby(['city_name', 'retention_segment'])['group_tc'].value_counts(True).unstack(fill_value=0)

group_tc                      control      test
city_name retention_segment                    
Bangalore DORMANT            0.697715  0.302285
          ELITE              0.700492  0.299508
          GOLD               0.700889  0.299111
          HH                 0.700191  0.299809
          INACTIVE           0.697882  0.302118
          PLATINUM           0.699515  0.300485
          PRIME              0.696201  0.303799
          SILVER             0.700654  0.299346
          UNKNOWN            0.697145  0.302855

### Export

In [26]:
df_final_sample.head(5)

,customer_id,city_name,customer_mobile,gross_count,net_count,cobrm,cobra,ocara,ltr_segment,retention_segment,lifetime_stage,service_affinity,gender,group_tc
870183,64fea074729cc433fea61a7e,Bangalore,7349006863,2,2,0,0,0,PHH,GOLD,SUSTENANCE,ONLY_AUTO,MALE,control
1596516,5d957409941a7d1c51d17de3,Bangalore,7209714014,1,0,0,0,1,PHH,DORMANT,DORMANT,UNKNOWN,FEMALE,control
1055439,66224e83fa9d7d3ce75f2baa,Bangalore,8095552517,3,2,0,1,0,PHH,GOLD,SUSTENANCE,ONLY_AUTO,FEMALE,control
1504823,5c62a327f2edc73367569755,Bangalore,9513418912,26,18,0,4,2,PHH,ELITE,COMMITTED,ONLY_AUTO,MALE,control
2053785,64deff0683f730aa7f40656a,Bangalore,9945630089,2,2,0,0,0,HH,HH,HANDHOLDING,UNKNOWN,FEMALE,control


In [27]:
# df_train_control['customer_mobile'] = df_train_control['customer_mobile'].astype(float).astype(int)

In [28]:
df_final_sample[df_final_sample['group_tc'] == 'test'].to_csv(local_datasource + 'test_group.csv', index=False)

In [29]:
df_export = df_final_sample[df_final_sample['group_tc'] == 'test']

In [30]:
df_export.shape

(619046, 14)

In [31]:
df_export.head(5)

,customer_id,city_name,customer_mobile,gross_count,net_count,cobrm,cobra,ocara,ltr_segment,retention_segment,lifetime_stage,service_affinity,gender,group_tc
1347341,64ca00efa175eb7ed66e4d1f,Bangalore,6374678431,1,1,0,0,0,PHH,GOLD,HOOK,UNKNOWN,MALE,test
1600767,59e040da5c873be956ace772,Bangalore,9844830292,12,8,0,2,2,PHH,ELITE,COMMITTED,ONLY_AUTO,FEMALE,test
691892,66c9e5acf5b376d01af93ed7,Bangalore,8660959016,2,2,0,0,0,HH,HH,HANDHOLDING,UNKNOWN,MALE,test
1477036,64829f21b5997b57a0118110,Bangalore,7996253243,3,1,0,1,0,PHH,GOLD,SUSTENANCE,ONLY_AUTO,FEMALE,test
593926,5b33dbbb50b4683179f5bf3f,Bangalore,9035906062,3,1,0,1,1,PHH,SILVER,HOOK,UNKNOWN,MALE,test


In [32]:
df_export[['customer_mobile']].to_csv(local_datasource + 'test_group_final.csv', header=False, index=False)

In [34]:
df_export[['customer_mobile']].iloc[:300000]

,customer_mobile
1347341,6374678431
1600767,9844830292
691892,8660959016
1477036,7996253243
593926,9035906062
...,...
1846680,9986947054
713641,9972954394
1377497,9566260919
1193309,8408089165


In [58]:
df_export[['customer_mobile']].iloc[:206340].to_csv(local_datasource + 'test_group_final1.csv', header=False, index=False)
df_export[['customer_mobile']].iloc[206340:412680].to_csv(local_datasource + 'test_group_final2.csv', header=False, index=False)
df_export[['customer_mobile']].iloc[412680:].to_csv(local_datasource + 'test_group_final3.csv', header=False, index=False)

In [49]:
test1 = df_export[['customer_mobile']].iloc[:206340]
test2 = df_export[['customer_mobile']].iloc[206340:412680]
test3 = df_export[['customer_mobile']].iloc[412680:]

In [57]:
pd.merge(test2,test3,how='inner', on='customer_mobile')

,customer_mobile


In [53]:
pd.merge(test3,test_merge,how='inner', on='customer_mobile') 

,customer_mobile


In [43]:
619046/3

206348.66666666666

In [48]:
206340 + 206340

412680

In [47]:
206340+412680

619020